# 第1C阶段：更强编码器 + 低秩协方差

## 背景
第1A/1B阶段使用 `all-MiniLM-L6-v2`（384维），各向异性方法在所有配置下检索F1均低于余弦基线1.6-2.1%。

**核心假设**：MiniLM 的 384 维嵌入空间语义分辨率不足——维度间缺乏足够的语义独立性，导致对角协方差没有足够的"杠杆"来差异化加权。

## 本 Notebook 的实验

**第一轮：更强编码器**
- E5-large（1024维）: 当前最强开源嵌入模型之一
- BGE-large（1024维）: 备选

**第二轮：低秩非对角协方差**（如果对角协方差仍不够）
- 将协方差矩阵参数化为 Σ = diag(d) + L·Lᵀ，其中 L 是 d×r 的低秩矩阵
- 这允许捕捉维度间的交互，而参数量仅增加 d×r

## 硬件
- T4（免费版）可运行（E5-large 约 1.3GB，编码速度较慢但可行）
- A100 可显著加速编码阶段

In [1]:
%%capture
!pip install transformers datasets sentence-transformers accelerate
!pip install bert-score rouge-score scipy scikit-learn matplotlib seaborn tqdm

In [ ]:
import os, json, random, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix', 'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.8, 'axes.grid': True, 'grid.alpha': 0.3,
})
print(f'设备: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

---
## 第1部分：数据准备（复用 HotpotQA）

In [ ]:
from datasets import load_dataset

print('正在加载 HotpotQA...')
hotpot = load_dataset('hotpot_qa', 'distractor', split='validation')

def parse_sample(s):
    sup = set(s['supporting_facts']['title'])
    ctxs = [{'title': t, 'text': ' '.join(sents), 'is_supporting': t in sup}
            for t, sents in zip(s['context']['title'], s['context']['sentences'])]
    return {'query': s['question'], 'answer': s['answer'], 'contexts': ctxs}

parsed = [parse_sample(s) for s in hotpot]
random.shuffle(parsed)
train_data = parsed[:2000]
val_data = parsed[2000:2200]
test_data = parsed[2200:2700]
print(f'训练: {len(train_data)}, 验证: {len(val_data)}, 测试: {len(test_data)}')

---
## 第2部分：多编码器对比编码

同时编码三种编码器，后续统一对比。

In [ ]:
from sentence_transformers import SentenceTransformer

def encode_all(data, encoder, batch_size=64):
    """编码所有查询和上下文段落"""
    queries = [d['query'] for d in data]
    all_ctx, ctx_map = [], []
    for i, d in enumerate(data):
        for j, c in enumerate(d['contexts']):
            all_ctx.append(c['text']); ctx_map.append((i, j))

    q_embs = encoder.encode(queries, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    c_embs = encoder.encode(all_ctx, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

    result = []
    for i, d in enumerate(data):
        s = dict(d); s['query_emb'] = q_embs[i]; s['context_embs'] = []; result.append(s)
    for (i, j), emb in zip(ctx_map, c_embs):
        result[i]['context_embs'].append(emb)
    for s in result:
        s['context_embs'] = np.array(s['context_embs'])
    return result

# 定义要测试的编码器
ENCODERS = {
    'MiniLM-384d': 'all-MiniLM-L6-v2',
    'E5-large-1024d': 'intfloat/e5-large-v2',
}

encoded_sets = {}  # {encoder_name: {'train': [...], 'val': [...], 'test': [...]}}

for enc_name, model_name in ENCODERS.items():
    print(f'\n{"="*60}')
    print(f'编码器: {enc_name} ({model_name})')
    print(f'{"="*60}')
    enc = SentenceTransformer(model_name, device=DEVICE)
    dim = enc.get_sentence_embedding_dimension()
    print(f'嵌入维度: {dim}')

    print('编码训练集...')
    tr = encode_all(train_data, enc)
    print('编码验证集...')
    va = encode_all(val_data, enc)
    print('编码测试集...')
    te = encode_all(test_data, enc)

    encoded_sets[enc_name] = {'train': tr, 'val': va, 'test': te, 'dim': dim}
    del enc  # 释放显存
    torch.cuda.empty_cache()

print('\n✅ 所有编码器编码完成')
for name, d in encoded_sets.items():
    print(f'  {name}: {d["dim"]}维, 训练{len(d["train"])}样本')

---
## 第3部分：核心模型定义

包含两种协方差参数化：
1. **对角协方差**（Diagonal）：每维度独立方差，参数量 = d
2. **低秩+对角协方差**（LowRank）：Σ = diag(d) + L·Lᵀ，参数量 = d + d×r

In [ ]:
# ====== 对角协方差模型（第1A/1B阶段的改进版）======

class DiagonalVariancePredictor(nn.Module):
    """对角协方差：每维度独立方差"""
    def __init__(self, embed_dim, hidden_dim=256, clamp_min=-3.0, clamp_max=3.0):
        super().__init__()
        self.clamp_min, self.clamp_max = clamp_min, clamp_max
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, embed_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, q):
        return torch.clamp(self.net(q), self.clamp_min, self.clamp_max)

    def log_density(self, q_emb, ctx_embs, log_var):
        mu = q_emb.unsqueeze(1)
        lv = log_var.unsqueeze(1)
        diff = ctx_embs - mu
        var = torch.exp(lv)
        mahal = torch.sum(diff**2 / var, dim=-1)
        log_norm = torch.sum(lv, dim=-1)
        return -0.5 * (mahal + log_norm)


# ====== 低秩+对角协方差模型 ======

class LowRankVariancePredictor(nn.Module):
    """
    低秩+对角协方差: Σ = diag(exp(log_d)) + L·Lᵀ
    - log_d: 对角部分的对数方差 (d,)
    - L: 低秩因子 (d, r)
    精度矩阵通过 Woodbury 公式高效计算。
    """
    def __init__(self, embed_dim, rank=16, hidden_dim=256, clamp_min=-3.0, clamp_max=3.0):
        super().__init__()
        self.rank = rank
        self.embed_dim = embed_dim
        self.clamp_min, self.clamp_max = clamp_min, clamp_max

        # 共享主干
        self.backbone = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(0.1),
        )
        # 对角部分
        self.head_diag = nn.Linear(hidden_dim, embed_dim)
        nn.init.zeros_(self.head_diag.weight); nn.init.zeros_(self.head_diag.bias)
        # 低秩部分
        self.head_lowrank = nn.Linear(hidden_dim, embed_dim * rank)
        nn.init.normal_(self.head_lowrank.weight, std=0.01)
        nn.init.zeros_(self.head_lowrank.bias)

    def forward(self, q):
        """返回 (log_diag, L_factor)"""
        h = self.backbone(q)  # (B, H)
        log_d = torch.clamp(self.head_diag(h), self.clamp_min, self.clamp_max)  # (B, D)
        L = self.head_lowrank(h).view(-1, self.embed_dim, self.rank) * 0.1  # (B, D, R), 缩小初始值
        return log_d, L

    def log_density(self, q_emb, ctx_embs, params):
        """
        计算 log N(x | mu, Σ)，其中 Σ = diag(d) + L·Lᵀ
        使用 Woodbury 公式高效求逆:
          Σ⁻¹ = D⁻¹ - D⁻¹·L·(I + Lᵀ·D⁻¹·L)⁻¹·Lᵀ·D⁻¹
        """
        log_d, L = params  # log_d: (B,D), L: (B,D,R)
        B, N, D = ctx_embs.shape
        R = self.rank

        d = torch.exp(log_d)  # (B, D) 对角方差
        d_inv = 1.0 / d       # (B, D)

        diff = ctx_embs - q_emb.unsqueeze(1)  # (B, N, D)

        # 对角部分: diff^T · D^{-1} · diff
        mahal_diag = torch.sum(diff**2 * d_inv.unsqueeze(1), dim=-1)  # (B, N)

        # Woodbury 修正项
        # M = I_r + L^T D^{-1} L, 大小 (B, R, R)
        DinvL = d_inv.unsqueeze(-1) * L  # (B, D, R)
        M = torch.eye(R, device=q_emb.device).unsqueeze(0) + torch.bmm(L.transpose(1,2), DinvL)  # (B, R, R)
        M_inv = torch.linalg.inv(M)  # (B, R, R)

        # v = D^{-1} · diff, 大小 (B, N, D)
        v = diff * d_inv.unsqueeze(1)
        # w = L^T · v^T = (B, R, N)
        w = torch.bmm(L.transpose(1,2), v.transpose(1,2))  # (B, R, N)
        # correction = w^T · M^{-1} · w 的对角元素
        Minv_w = torch.bmm(M_inv, w)  # (B, R, N)
        mahal_correction = torch.sum(w * Minv_w, dim=1)  # (B, N)

        mahal = mahal_diag - mahal_correction  # (B, N)

        # log |Σ| = log |D| + log |M| = sum(log_d) + log|M|
        log_det_D = torch.sum(log_d, dim=-1, keepdim=True)  # (B, 1)
        log_det_M = torch.linalg.slogdet(M)[1].unsqueeze(-1)  # (B, 1)
        log_det = log_det_D + log_det_M  # (B, 1)

        return -0.5 * (mahal + log_det)  # (B, N)


print('✅ 模型定义完成: DiagonalVariancePredictor, LowRankVariancePredictor')

In [ ]:
# ====== 公共工具函数 ======

class CtxDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        s = self.data[i]
        return (torch.tensor(s['query_emb'], dtype=torch.float32),
                torch.tensor(s['context_embs'], dtype=torch.float32),
                torch.tensor([1.0 if c['is_supporting'] else 0.0 for c in s['contexts']], dtype=torch.float32))


def infonce(log_density, labels, temperature=0.1):
    """InfoNCE 损失"""
    B, N = log_density.shape
    pos_mask, neg_mask = labels.bool(), ~labels.bool()
    loss = torch.tensor(0.0, device=log_density.device)
    n = 0
    for b in range(B):
        ps, ns = log_density[b][pos_mask[b]], log_density[b][neg_mask[b]]
        if len(ps) == 0 or len(ns) == 0: continue
        for p in ps:
            logits = torch.cat([p.unsqueeze(0), ns]) / temperature
            loss += nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([0], device=logits.device))
            n += 1
    return loss / max(n, 1)


def select_cosine(q, c, k=2):
    sims = cosine_similarity(q.reshape(1,-1), c)[0]
    return np.argsort(sims)[-k:][::-1], sims

def select_random(q, c, k=2):
    return np.random.choice(len(c), k, replace=False), np.zeros(len(c))


def eval_f1(data, select_fn, k=2, **kw):
    f1s = []
    for s in data:
        sel, _ = select_fn(s['query_emb'], s['context_embs'], k=k, **kw)
        labels = [c['is_supporting'] for c in s['contexts']]
        hit = sum(1 for i in sel if labels[i])
        p = hit/k; r = hit/max(sum(labels),1)
        f1s.append(2*p*r/(p+r) if (p+r)>0 else 0)
    return np.mean(f1s)


def compute_cv(model, data, is_lowrank=False):
    model.eval()
    all_vars = []
    with torch.no_grad():
        for s in data[:100]:
            q = torch.tensor(s['query_emb'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            if is_lowrank:
                log_d, L = model(q)
                # 有效方差 = diag(Σ) = exp(log_d) + sum(L^2, dim=-1)
                eff_var = torch.exp(log_d) + (L**2).sum(dim=-1)
                all_vars.append(eff_var.cpu().numpy()[0])
            else:
                lv = model(q).cpu().numpy()[0]
                all_vars.append(np.exp(lv))
    v = np.array(all_vars)
    cv = np.mean(np.std(v, axis=1) / (np.mean(v, axis=1) + 1e-8))
    mv = np.mean(v, axis=0)
    ratio = np.max(mv) / (np.min(mv) + 1e-8)
    return cv, ratio

print('✅ 工具函数就绪')

In [ ]:
# ====== 统一训练函数（支持对角和低秩）======

def train_experiment(name, enc_name, model_type='diagonal', rank=16,
                     lr=5e-4, epochs=30, temperature=0.1, clamp=(-3, 3)):
    """
    训练实验。
    model_type: 'diagonal' 或 'lowrank'
    """
    data = encoded_sets[enc_name]
    dim = data['dim']
    is_lr = (model_type == 'lowrank')

    print(f'\n{"="*70}')
    print(f'实验: {name}')
    print(f'  编码器: {enc_name} ({dim}维)')
    print(f'  协方差: {model_type}' + (f' (rank={rank})' if is_lr else ''))
    print(f'  log_var 范围: [{clamp[0]}, {clamp[1]}]')
    print(f'{"="*70}')

    if is_lr:
        model = LowRankVariancePredictor(dim, rank=rank, clamp_min=clamp[0], clamp_max=clamp[1]).to(DEVICE)
    else:
        model = DiagonalVariancePredictor(dim, clamp_min=clamp[0], clamp_max=clamp[1]).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  参数量: {n_params:,}')

    train_loader = DataLoader(CtxDataset(data['train']), batch_size=32, shuffle=True)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1, best_state = -1, None
    hist = defaultdict(list)

    for ep in range(epochs):
        model.train()
        ep_loss, nb = 0, 0
        for q, c, l in train_loader:
            q, c, l = q.to(DEVICE), c.to(DEVICE), l.to(DEVICE)
            if is_lr:
                params = model(q)
                ld = model.log_density(q, c, params)
            else:
                lv = model(q)
                ld = model.log_density(q, c, lv)
            loss = infonce(ld, l, temperature)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item(); nb += 1
        scheduler.step()

        # 验证
        def _sel(q, c, k=2):
            model.eval()
            with torch.no_grad():
                qt = torch.tensor(q, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                ct = torch.tensor(c, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                if is_lr:
                    p = model(qt)
                    sc = model.log_density(qt, ct, p)[0].cpu().numpy()
                else:
                    lv = model(qt)
                    sc = model.log_density(qt, ct, lv)[0].cpu().numpy()
            return np.argsort(sc)[-k:][::-1], sc

        vf1 = eval_f1(data['val'], _sel, k=2)
        hist['loss'].append(ep_loss/nb)
        hist['val_f1'].append(vf1)

        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (ep+1) % 10 == 0 or ep == 0:
            cv, ratio = compute_cv(model, data['val'], is_lr)
            print(f'  轮次 {ep+1:3d}/{epochs} | 损失: {ep_loss/nb:.4f} | '
                  f'验证F1: {vf1:.4f} | CV: {cv:.3f} | 方差比: {ratio:.1f}x')

    model.load_state_dict(best_state)
    print(f'  ✅ 最优验证F1: {best_f1:.4f}')
    return model, dict(hist), is_lr

print('✅ 训练函数就绪')

---
## 第4部分：第一轮实验——编码器对比

在两种编码器上分别训练对角协方差模型，使用第1B阶段验证过的最优配置（InfoNCE + 截断[-3,3]）。

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Redefine infonce to handle attention mask
def infonce(log_density, labels, attention_mask, temperature=0.1):
    """InfoNCE 损失，支持attention mask"""
    B, N = log_density.shape
    # Mask out padded log_density values to -inf
    log_density_masked = log_density.masked_fill(~attention_mask, float('-inf'))
    # Mask out labels for padded items (0.0 will be False for bool())
    labels_masked = labels.masked_fill(~attention_mask, 0.0)

    loss = torch.tensor(0.0, device=log_density.device)
    n = 0
    for b in range(B):
        # Filter for actual positive and negative samples, ignoring padded entries
        ps_all = log_density_masked[b][labels_masked[b].bool()]
        ns_all = log_density_masked[b][~labels_masked[b].bool()]

        ps = ps_all[ps_all != float('-inf')]
        ns = ns_all[ns_all != float('-inf')]

        if len(ps) == 0 or len(ns) == 0:
            continue

        for p in ps:
            logits = torch.cat([p.unsqueeze(0), ns]) / temperature
            loss += nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([0], device=logits.device))
            n += 1
    return loss / max(n, 1)

# Redefine train_experiment with custom collate_fn
def train_experiment(name, enc_name, model_type='diagonal', rank=16,
                     lr=5e-4, epochs=30, temperature=0.1, clamp=(-3, 3)):
    """
    训练实验。
    model_type: 'diagonal' 或 'lowrank'
    """
    data = encoded_sets[enc_name]
    dim = data['dim']
    is_lr = (model_type == 'lowrank')

    print(f'\n{"="*70}')
    print(f'实验: {name}')
    print(f'  编码器: {enc_name} ({dim}维)')
    print(f'  协方差: {model_type}' + (f' (rank={rank})' if is_lr else ''))
    print(f'  log_var 范围: [{clamp[0]}, {clamp[1]}]')
    print(f'{"="*70}')

    if is_lr:
        model = LowRankVariancePredictor(dim, rank=rank, clamp_min=clamp[0], clamp_max=clamp[1]).to(DEVICE)
    else:
        model = DiagonalVariancePredictor(dim, clamp_min=clamp[0], clamp_max=clamp[1]).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  参数量: {n_params:,}')

    # Custom collate function for DataLoader
    def custom_collate_fn(batch):
        query_embs = torch.stack([s[0] for s in batch])
        max_num_ctx = max(s[1].shape[0] for s in batch)
        embed_dim = batch[0][1].shape[1]

        padded_context_embs = torch.zeros(len(batch), max_num_ctx, embed_dim, dtype=torch.float32)
        padded_labels = torch.zeros(len(batch), max_num_ctx, dtype=torch.float32)
        attention_mask = torch.zeros(len(batch), max_num_ctx, dtype=torch.bool) # Mask for valid contexts

        for i, (q_emb, ctx_embs, labels) in enumerate(batch):
            num_ctx = ctx_embs.shape[0]
            padded_context_embs[i, :num_ctx] = ctx_embs
            padded_labels[i, :num_ctx] = labels
            attention_mask[i, :num_ctx] = True

        return query_embs, padded_context_embs, padded_labels, attention_mask

    train_loader = DataLoader(CtxDataset(data['train']), batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1, best_state = -1, None
    hist = defaultdict(list)

    for ep in range(epochs):
        model.train()
        ep_loss, nb = 0, 0
        for q, c, l, mask in train_loader: # Unpack attention_mask here
            q, c, l, mask = q.to(DEVICE), c.to(DEVICE), l.to(DEVICE), mask.to(DEVICE)
            if is_lr:
                params = model(q)
                ld = model.log_density(q, c, params)
            else:
                lv = model(q)
                ld = model.log_density(q, c, lv)
            loss = infonce(ld, l, mask, temperature) # Pass mask to infonce
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item(); nb += 1
        scheduler.step()

        # 验证
        def _sel(q_single, c_single, k=2): # Renamed q, c to q_single, c_single to avoid confusion
            model.eval()
            with torch.no_grad():
                qt = torch.tensor(q_single, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                ct = torch.tensor(c_single, dtype=torch.float32).unsqueeze(0).to(DEVICE)
                if is_lr:
                    p = model(qt)
                    sc = model.log_density(qt, ct, p)[0].cpu().numpy()
                else:
                    lv = model(qt)
                    sc = model.log_density(qt, ct, lv)[0].cpu().numpy()
            return np.argsort(sc)[-k:][::-1], sc

        vf1 = eval_f1(data['val'], _sel, k=2)
        hist['loss'].append(ep_loss/nb)
        hist['val_f1'].append(vf1)

        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (ep+1) % 10 == 0 or ep == 0:
            cv, ratio = compute_cv(model, data['val'], is_lr)
            print(f'  轮次 {ep+1:3d}/{epochs} | 损失: {ep_loss/nb:.4f} | '
                  f'验证F1: {vf1:.4f} | CV: {cv:.3f} | 方差比: {ratio:.1f}x')

    model.load_state_dict(best_state)
    print(f'  ✅ 最优验证F1: {best_f1:.4f}')
    return model, dict(hist), is_lr


# ====== 实验1: MiniLM + 对角（复现第1B的实验D作为基线）======
model_mini_diag, hist_mini_diag, _ = train_experiment(
    'MiniLM + 对角', 'MiniLM-384d', model_type='diagonal', epochs=30)


In [ ]:
# ====== 实验2: E5-large + 对角 ======
model_e5_diag, hist_e5_diag, _ = train_experiment(
    'E5-large + 对角', 'E5-large-1024d', model_type='diagonal', epochs=30)

---
## 第5部分：第二轮实验——低秩协方差

在最强编码器上测试低秩非对角协方差。
测试 rank=8, 16, 32 三种配置。

In [ ]:
# ====== 实验3: E5-large + 低秩 rank=8 ======
model_e5_lr8, hist_e5_lr8, _ = train_experiment(
    'E5-large + 低秩(r=8)', 'E5-large-1024d', model_type='lowrank', rank=8, epochs=30)

In [ ]:
# ====== 实验4: E5-large + 低秩 rank=16 ======
model_e5_lr16, hist_e5_lr16, _ = train_experiment(
    'E5-large + 低秩(r=16)', 'E5-large-1024d', model_type='lowrank', rank=16, epochs=30)

In [ ]:
# ====== 实验5: E5-large + 低秩 rank=32 ======
model_e5_lr32, hist_e5_lr32, _ = train_experiment(
    'E5-large + 低秩(r=32)', 'E5-large-1024d', model_type='lowrank', rank=32, epochs=30)

---
## 第6部分：全面对比评估

In [ ]:
# ====== 测试集全面评估 ======

def make_selector(model, is_lowrank=False):
    def _sel(q, c, k=2):
        model.eval()
        with torch.no_grad():
            qt = torch.tensor(q, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            ct = torch.tensor(c, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            if is_lowrank:
                p = model(qt)
                sc = model.log_density(qt, ct, p)[0].cpu().numpy()
            else:
                lv = model(qt)
                sc = model.log_density(qt, ct, lv)[0].cpu().numpy()
        return np.argsort(sc)[-k:][::-1], sc
    return _sel

experiments = [
    ('余弦 Top-K (MiniLM)', 'MiniLM-384d', None, False),
    ('余弦 Top-K (E5-large)', 'E5-large-1024d', None, False),
    ('MiniLM + 对角', 'MiniLM-384d', model_mini_diag, False),
    ('E5-large + 对角', 'E5-large-1024d', model_e5_diag, False),
    ('E5-large + 低秩(r=8)', 'E5-large-1024d', model_e5_lr8, True),
    ('E5-large + 低秩(r=16)', 'E5-large-1024d', model_e5_lr16, True),
    ('E5-large + 低秩(r=32)', 'E5-large-1024d', model_e5_lr32, True),
]

print('='*85)
print('测试集完整评估')
print('='*85)
print(f'{"方法":30s} | {"F1(k=2)":>8s} | {"F1(k=3)":>8s} | {"F1(k=4)":>8s} | {"CV":>8s} | {"vs对应余弦":>12s}')
print('-'*85)

all_results = {}
cos_baselines = {}

for name, enc, model, is_lr in experiments:
    test_d = encoded_sets[enc]['test']
    val_d = encoded_sets[enc]['val']

    if model is None:  # 余弦基线
        f1s = {k: eval_f1(test_d, select_cosine, k=k) for k in [2,3,4]}
        cos_baselines[enc] = f1s[2]
        cv_str = 'N/A'
        delta_str = '---'
    else:
        sel = make_selector(model, is_lr)
        f1s = {k: eval_f1(test_d, sel, k=k) for k in [2,3,4]}
        cv, ratio = compute_cv(model, val_d, is_lr)
        cv_str = f'{cv:.3f}'
        base = cos_baselines.get(enc, 0)
        d = f1s[2] - base
        marker = '✅' if d > 0.005 else '⚡' if d > -0.005 else '❌'
        delta_str = f'{d:+.4f} {marker}'

    all_results[name] = f1s
    print(f'{name:30s} | {f1s[2]:8.4f} | {f1s[3]:8.4f} | {f1s[4]:8.4f} | {cv_str:>8s} | {delta_str:>12s}')

print('='*85)

In [ ]:
# ====== 结果可视化 ======

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

# (a) F1 对比柱状图
methods = list(all_results.keys())
f1_vals = [all_results[m][2] for m in methods]
colors = ['#BDBDBD', '#969696',  # 余弦基线
          '#FD8D3C',  # MiniLM对角
          '#2171B5',  # E5对角
          '#6BAED6', '#2171B5', '#08519C']  # E5低秩
short_names = ['Cos\nMiniLM', 'Cos\nE5', 'MiniLM\nDiag',
               'E5\nDiag', 'E5\nLR-8', 'E5\nLR-16', 'E5\nLR-32']

bars = axes[0].bar(range(len(methods)), f1_vals, color=colors, alpha=0.9,
                   edgecolor='#333', linewidth=0.5)
for b, v in zip(bars, f1_vals):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                 f'{v:.3f}', ha='center', fontsize=6.5)
axes[0].set_xticks(range(len(methods)))
axes[0].set_xticklabels(short_names, fontsize=7)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Retrieval F1 (k=2)')
axes[0].set_ylim(0, 1)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# (b) 训练曲线对比
hist_items = [
    ('MiniLM Diag', hist_mini_diag, '#FD8D3C'),
    ('E5 Diag', hist_e5_diag, '#2171B5'),
    ('E5 LR-8', hist_e5_lr8, '#6BAED6'),
    ('E5 LR-16', hist_e5_lr16, '#2171B5'),
    ('E5 LR-32', hist_e5_lr32, '#08519C'),
]
for label, h, c in hist_items:
    axes[1].plot(h['val_f1'], color=c, label=label, lw=1.2)

# 画余弦基线
for enc, c, ls in [('MiniLM-384d', '#BDBDBD', '--'), ('E5-large-1024d', '#969696', '-')]:
    axes[1].axhline(y=cos_baselines.get(enc, 0), color=c, ls=ls, lw=1, label=f'Cos {enc[:5]}')

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val F1')
axes[1].set_title('Validation F1 over Training')
axes[1].legend(fontsize=6, ncol=2)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('phase1c_comparison.pdf', dpi=300)
plt.savefig('phase1c_comparison.png', dpi=300)
plt.show()
print('📊 已保存: phase1c_comparison.pdf / .png')

---
## 第7部分：总结与判定

In [ ]:
# ====== 最终总结 ======

print('\n' + '#'*70)
print('#  第1C阶段实验总结')
print('#'*70)

print('\n## 各方法 F1 (k=2) 汇总')
for name, f1s in all_results.items():
    enc = 'E5-large-1024d' if 'E5' in name else 'MiniLM-384d'
    base = cos_baselines.get(enc, 0)
    d = f1s[2] - base
    marker = '✅' if d > 0.005 else '⚡' if d > -0.005 else '❌'
    print(f'  {marker} {name:30s} F1={f1s[2]:.4f} (Δ={d:+.4f})')

# 找出最优的非基线方法
non_baseline = {n: r for n, r in all_results.items() if '余弦' not in n}
best_name = max(non_baseline, key=lambda n: non_baseline[n][2])
best_f1 = non_baseline[best_name][2]
best_enc = 'E5-large-1024d' if 'E5' in best_name else 'MiniLM-384d'
best_base = cos_baselines[best_enc]
best_delta = best_f1 - best_base

print(f'\n## 最优模型: {best_name}')
print(f'  F1={best_f1:.4f}, vs 对应余弦基线 Δ={best_delta:+.4f}')

print(f'\n## 关键问题回答')

# Q1: 更强编码器是否有帮助？
e5_diag_f1 = all_results.get('E5-large + 对角', {}).get(2, 0)
mini_diag_f1 = all_results.get('MiniLM + 对角', {}).get(2, 0)
e5_cos = cos_baselines.get('E5-large-1024d', 0)
mini_cos = cos_baselines.get('MiniLM-384d', 0)

e5_delta = e5_diag_f1 - e5_cos
mini_delta = mini_diag_f1 - mini_cos

print(f'  Q1: 更强编码器是否缩小了与余弦基线的差距？')
print(f'      MiniLM 各向异性 vs 余弦: Δ={mini_delta:+.4f}')
print(f'      E5     各向异性 vs 余弦: Δ={e5_delta:+.4f}')
if e5_delta > mini_delta + 0.005:
    print(f'      ✅ 是，E5 下差距更小（或已超越）')
elif e5_delta > mini_delta - 0.005:
    print(f'      ⚡ 差距相近，编码器不是瓶颈')
else:
    print(f'      ❌ 否，更强编码器反而差距更大')

# Q2: 低秩协方差是否有帮助？
lr_results = {n: r[2] for n, r in all_results.items() if '低秩' in n}
if lr_results:
    best_lr = max(lr_results, key=lr_results.get)
    best_lr_f1 = lr_results[best_lr]
    lr_vs_diag = best_lr_f1 - e5_diag_f1
    lr_vs_cos = best_lr_f1 - e5_cos
    print(f'\n  Q2: 低秩协方差是否优于对角？')
    print(f'      最优低秩: {best_lr} F1={best_lr_f1:.4f}')
    print(f'      vs E5对角: Δ={lr_vs_diag:+.4f}')
    print(f'      vs E5余弦: Δ={lr_vs_cos:+.4f}')
    if lr_vs_cos > 0.005:
        print(f'      ✅ 低秩协方差超越了余弦基线！')
    elif lr_vs_diag > 0.005:
        print(f'      ⚡ 低秩优于对角，但仍未超越余弦')
    else:
        print(f'      ❌ 低秩未带来显著提升')

print(f'\n## 下一步建议')
if best_delta > 0.005:
    print('  ✅ 成功！可以开始撰写论文。')
    print('  建议补充: 在最优配置上运行生成质量评估（ROUGE/BERTScore）')
elif best_delta > -0.01:
    print('  ⚡ 接近平局。结合生成质量提升，仍有论文价值。')
    print('  建议: 运行生成质量评估确认ROUGE提升是否保持')
    print('  建议: 尝试基于注意力的软区域选择作为替代方案')
else:
    print('  ❌ 各向异性高斯区域方法在检索F1上未能超越余弦基线。')
    print('  建议: 以负面结果+生成质量矛盾为核心撰写分析论文')
    print('  建议: 探索基于注意力的非参数化区域选择')

print('\n' + '#'*70)

In [ ]:
# ====== 保存所有结果 ======

checkpoint = {
    'all_results': {n: {str(k): v for k, v in r.items()} for n, r in all_results.items()},
    'cos_baselines': cos_baselines,
    'best_method': best_name,
    'best_f1': best_f1,
    'best_delta': best_delta,
}

# 保存最优模型
best_model_map = {
    'MiniLM + 对角': model_mini_diag,
    'E5-large + 对角': model_e5_diag,
    'E5-large + 低秩(r=8)': model_e5_lr8,
    'E5-large + 低秩(r=16)': model_e5_lr16,
    'E5-large + 低秩(r=32)': model_e5_lr32,
}
if best_name in best_model_map:
    checkpoint['model_state_dict'] = best_model_map[best_name].state_dict()

torch.save(checkpoint, 'phase1c_results.pt')
print('💾 已保存: phase1c_results.pt')
print('   包含所有实验结果和最优模型参数。')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')